# **Implementing a Matrix Factorization-based Recommender System**

## **Represent user and item by Matrix Factorization**
 - Users and items are represented through matrix factorization.
  - A user-item interaction matrix $( R \in \mathbb{R}^{n \times m})$ is approximated as the product of two matrices: $( R \approx P \times Q)$, where $( P \in \mathbb{R}^{n \times d})$ and $( Q \in \mathbb{R}^{m \times d})$.
  - $ n $ is the number of users, $ m $ is the number of items, and $ d $ is the dimension of the embedding vectors.

**How to do Matrix Factorization**:
   - The goal is to find a good representation for users and items.
   - The objective is to minimize the differences between the predicted and actual interaction values: $ \min_{P,Q} \sum_{(u,i) \in R'} (r_{ui} - P_u Q_i)^2 $.
   - Not all elements in $ R $ are known; $ R' $ is the set of known elements in $ R $.
   - $ r_{ui} $ is the interaction record of user $ u $ and item $ i $.
   - $ P_u $ is the embedding vector for user $ u $, and $ Q_i $ is the embedding vector for item $ i $.
   - The interaction probability between user $ u $ and item $ i $ is $ r_{ui} = P_u Q_i $.

## **Requirements:**
In this practice, you will implement a recommender system using **Matrix Factorization**.
You should:
   - Construct a matrix factorization-based recommender system using the positive data `train_pos.npy` provided in project 3.
   - For each user-item pair $ u, i $ in `train_pos.npy`, $ R_{ui} = 1 $.
   - If a user-item pair $ u^*, i^* $ is not in `train_pos.npy`, $ R_{u^*i^*} = 0 $.
   - The task is to find a good embedding representation for each user and item.


## **Reference Workflow**:
   1. Load the data and construct an interaction matrix.
   2. Obtain the embedding representation for each user and item.
      - **Use the objective function above and optimize the embeddings via gradient descent.**
      - **Note: The number of negative samples is much larger than that of positive samples. You can sample some negative samples in each iteration instead of using all negative samples.**
   3. Validate the effectiveness of the model.

### **Deadline:** 5.20



## **1 Load and Explore the Dataset**

In [16]:
import numpy as np
import random

# Load the dataset
train_pos = np.load("train_pos.npy")  # Contains user-item pairs
users, items = set(train_pos[:, 0]), set(train_pos[:, 1])
len(train_pos),train_pos[:5],train_pos[-5:]

(26638,
 array([[   0, 1113,    1],
        [   0,  736,    1],
        [   0,  888,    1],
        [   0,  636,    1],
        [   1,  374,    1]]),
 array([[6014,  934,    1],
        [6014, 1960,    1],
        [6014,  937,    1],
        [6014, 1963,    1],
        [6014, 1485,    1]]))

In [17]:
n_user, n_item = max(users) + 1, max(items) + 1
n_user, n_item

(np.int64(6015), np.int64(2347))

## **2. Initialize Parameters**

Initialize the embedding matrices $P$ for users and $Q$ for items. These matrices represent the user and item embeddings.

**Fill in the missing parts:**


In [18]:
# Define the embedding dimension
dim = 16  # Specify the embedding dimension

# Initialize user and item embeddings with random values
np.random.seed(42)
random.seed(42)
P = np.random.normal(loc=0.0, scale=0.1, size=(n_user, dim))  # user embeddings
Q = np.random.normal(loc=0.0, scale=0.1, size=(n_item, dim))  # item embeddings
P.shape, Q.shape

((6015, 16), (2347, 16))

## **3. Optimize the embeddings via gradient descent**

The loss function to optimize is Mean Squared Error (MSE):
$$
\text{Loss} = \sum_{(u, i) \in R'} (r_{ui} - P_u Q_i^T)^2
$$

OR add the regularization term:

$$
\text{Loss} = \sum_{(u, i) \in R'} (r_{ui} - P_u Q_i^T)^2 + \lambda (\|P_u\|^2 + \|Q_i\|^2)
$$

Here:
- $ R' $ is the set of the known elements in the $ R $
- $ r_{ui} $ is 1 for positive samples and 0 for negative samples.
- $ \lambda $ is the regularization term to prevent overfitting.


In [19]:
all_positive_pairs = train_pos[:, :2].astype(np.int64)
all_positive_set = set(map(tuple, all_positive_pairs))

test_ratio = 0.2
split_indices = np.random.permutation(len(all_positive_pairs))
test_size = int(len(all_positive_pairs) * test_ratio)
test_positive_pairs = all_positive_pairs[split_indices[:test_size]]
train_positive_pairs = all_positive_pairs[split_indices[test_size:]]

def sample_negative_pairs(num_samples):
    neg_users = np.random.randint(0, n_user, size=num_samples)
    neg_items = np.random.randint(0, n_item, size=num_samples)

    for idx in range(num_samples):
        while (int(neg_users[idx]), int(neg_items[idx])) in all_positive_set:
            neg_users[idx] = np.random.randint(0, n_user)
            neg_items[idx] = np.random.randint(0, n_item)

    return np.column_stack((neg_users, neg_items)).astype(np.int64)

learning_rate = 0.04
reg = 0.01
epochs = 30
negative_ratio = 1

for epoch in range(1, epochs + 1):
    negative_pairs = sample_negative_pairs(len(train_positive_pairs) * negative_ratio)
    epoch_pairs = np.vstack([train_positive_pairs, negative_pairs])
    epoch_ratings = np.concatenate([
        np.ones(len(train_positive_pairs), dtype=np.float64),
        np.zeros(len(negative_pairs), dtype=np.float64),
    ])

    order = np.random.permutation(len(epoch_pairs))
    total_loss = 0.0

    for sample_idx in order:
        u, i = epoch_pairs[sample_idx]
        rating = epoch_ratings[sample_idx]

        p_old = P[u].copy()
        q_old = Q[i].copy()
        prediction = float(np.dot(p_old, q_old))
        error = rating - prediction

        total_loss += error ** 2 + reg * (np.dot(p_old, p_old) + np.dot(q_old, q_old))
        P[u] += learning_rate * (error * q_old - reg * p_old)
        Q[i] += learning_rate * (error * p_old - reg * q_old)

    avg_loss = total_loss / len(epoch_pairs)
    print(f"Epoch {epoch:02d}/{epochs}, training loss: {avg_loss:.6f}")

Epoch 01/30, training loss: 0.504164
Epoch 02/30, training loss: 0.492273
Epoch 03/30, training loss: 0.475602
Epoch 04/30, training loss: 0.443462
Epoch 05/30, training loss: 0.388033
Epoch 06/30, training loss: 0.325810
Epoch 07/30, training loss: 0.273045
Epoch 08/30, training loss: 0.231937
Epoch 09/30, training loss: 0.200089
Epoch 10/30, training loss: 0.175840
Epoch 11/30, training loss: 0.156104
Epoch 12/30, training loss: 0.140705
Epoch 13/30, training loss: 0.127681
Epoch 14/30, training loss: 0.116785
Epoch 15/30, training loss: 0.107618
Epoch 16/30, training loss: 0.100354
Epoch 17/30, training loss: 0.094744
Epoch 18/30, training loss: 0.089361
Epoch 19/30, training loss: 0.084865
Epoch 20/30, training loss: 0.081401
Epoch 21/30, training loss: 0.078535
Epoch 22/30, training loss: 0.076467
Epoch 23/30, training loss: 0.074200
Epoch 24/30, training loss: 0.072222
Epoch 25/30, training loss: 0.070468
Epoch 26/30, training loss: 0.069360
Epoch 27/30, training loss: 0.067814
E

## **4 Verification**

Choose an appropriate metric to evaluate the results.

In [20]:
eval_negative_pairs = sample_negative_pairs(len(test_positive_pairs))
eval_pairs = np.vstack([test_positive_pairs, eval_negative_pairs])
eval_ratings = np.concatenate([
    np.ones(len(test_positive_pairs), dtype=np.float64),
    np.zeros(len(eval_negative_pairs), dtype=np.float64),
])

eval_scores = np.sum(P[eval_pairs[:, 0]] * Q[eval_pairs[:, 1]], axis=1)
mse = np.mean((eval_ratings - eval_scores) ** 2)
eval_predictions = (eval_scores >= 0.5).astype(np.float64)
accuracy = np.mean(eval_predictions == eval_ratings)

print(f"Test MSE: {mse:.6f}")
print(f"Test Accuracy: {accuracy:.6f}")

Test MSE: 0.306659
Test Accuracy: 0.653651
